In [19]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model

In [2]:
Textual_Data = pd.read_csv("Dataset/Non_Invasive_Anemia_Dataset.csv")  
Textual_Data.head()

,Subject_ID,Image_ID_Conjunctiva,Image_ID_Palm,Image_ID_Nail,HB_LEVEL,SEVERITY,Age(Months),Age (Years),GENDER,TYPE,COUNTRY,Data_Type
0,ID0001,Image_0001_C,Image_0001_P,Image_0001_N,9.8,Moderate,6,0.5,Female,Anemic,Ghana,Train
1,ID0002,Image_0002_C,Image_0002_P,Image_0002_N,9.9,Moderate,24,2.0,Male,Anemic,Ghana,Train
2,ID0003,Image_0003_C,Image_0003_P,Image_0003_N,11.1,Non-Anemic,24,2.0,Female,Non-Anemic,Ghana,Train
3,ID0004,Image_0004_C,Image_0004_P,Image_0004_N,12.5,Non-Anemic,12,1.0,Male,Non-Anemic,Ghana,Train
4,ID0005,Image_0005_C,Image_0005_P,Image_0005_N,9.9,Moderate,24,2.0,Male,Anemic,Ghana,Train


In [6]:
Textual_Data.shape

(1607, 13)

In [7]:
Textual_Data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1607 entries, 0 to 1606
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Subject_ID            1607 non-null   object 
 1   Image_ID_Conjunctiva  710 non-null    object 
 2   Image_ID_Palm         1607 non-null   object 
 3   Image_ID_Nail         1607 non-null   object 
 4   HB_LEVEL              1607 non-null   float64
 5   SEVERITY              1607 non-null   object 
 6   Age(Months)           1607 non-null   int64  
 7   Age (Years)           1607 non-null   float64
 8   GENDER                1607 non-null   object 
 9   TYPE                  1607 non-null   object 
 10  COUNTRY               1607 non-null   object 
 11  Data_Type             710 non-null    object 
 12  gender_encoded        1607 non-null   int64  
dtypes: float64(2), int64(2), object(9)
memory usage: 163.3+ KB


# Data Preprocessing

In [ ]:
le = LabelEncoder()
Textual_Data['gender_encoded'] = le.fit_transform(Textual_Data['GENDER'])  # Female=0, Male=1

In [4]:
Textual_Data.head()

,Subject_ID,Image_ID_Conjunctiva,Image_ID_Palm,Image_ID_Nail,HB_LEVEL,SEVERITY,Age(Months),Age (Years),GENDER,TYPE,COUNTRY,Data_Type,gender_encoded
0,ID0001,Image_0001_C,Image_0001_P,Image_0001_N,9.8,Moderate,6,0.5,Female,Anemic,Ghana,Train,0
1,ID0002,Image_0002_C,Image_0002_P,Image_0002_N,9.9,Moderate,24,2.0,Male,Anemic,Ghana,Train,1
2,ID0003,Image_0003_C,Image_0003_P,Image_0003_N,11.1,Non-Anemic,24,2.0,Female,Non-Anemic,Ghana,Train,0
3,ID0004,Image_0004_C,Image_0004_P,Image_0004_N,12.5,Non-Anemic,12,1.0,Male,Non-Anemic,Ghana,Train,1
4,ID0005,Image_0005_C,Image_0005_P,Image_0005_N,9.9,Moderate,24,2.0,Male,Anemic,Ghana,Train,1


In [8]:
scaler = StandardScaler()
Textual_Data['Scaled_Age'] = scaler.fit_transform(Textual_Data[['Age (Years)']])

In [10]:
Train_Set = Textual_Data[Textual_Data['Data_Type'] == 'Train']
Validation_Set = Textual_Data[Textual_Data['Data_Type'] == 'Val']
Test_Set = Textual_Data[Textual_Data['Data_Type'] == 'Test']

In [11]:
Train_Set.shape, Validation_Set.shape, Test_Set.shape

((496, 14), (105, 14), (109, 14))

In [12]:
X_text_train = Train_Set[['Scaled_Age', 'gender_encoded']].values
X_text_val   = Validation_Set[['Scaled_Age', 'gender_encoded']].values
X_text_test  = Test_Set[['Scaled_Age', 'gender_encoded']].values

In [13]:
X_text_train.shape, X_text_val.shape, X_text_test.shape

((496, 2), (105, 2), (109, 2))

# MLP Model

In [17]:
text_input = Input(shape=(2,), name="Text_Input")

x = Dense(16, activation='relu')(text_input)
x = Dense(32, activation='relu')(x)

text_features = Dense(32, activation='relu', name="Text_Features")(x)

text_mlp = Model(inputs=text_input, outputs=text_features)
text_mlp.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Text_Input (InputLayer)         │ (None, 2)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │            48 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Text_Features (Dense)           │ (None, 32)             │         1,056 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,648 (6.44 KB)

 Trainable params: 1,648 (6.44 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
text_train = text_mlp.predict(X_text_train)
text_val   = text_mlp.predict(X_text_val)
text_test  = text_mlp.predict(X_text_test)

16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 


In [21]:
np.save("Features/text_train.npy", text_train)
np.save("Features/text_val.npy", text_val)
np.save("Features/text_test.npy", text_test)